# Detecting Citation Manipulation via Neighborhood Label Entropy (NLE)

## Overview

This notebook demonstrates a novel graph-based method for detecting suspicious citation patterns in academic networks. 

**Method**: Neighborhood Label Entropy (NLE) - measures the diversity of class labels in a node's citation neighborhood. Low entropy may indicate:
- **Citation rings**: Groups of authors citing each other's work disproportionately
- **Selective citation**: Artificially boosting certain papers via coordinated citation
- **Homophily anomalies**: Unusual clustering of similar papers

**Dataset**: Cora citation network (2708 papers, 10556 edges, 7 classes)

**Output**: NLE scores for each node, visualization of suspicious patterns

In [ ]:
# Install dependencies (Colab-compatible pattern from aii-colab skill)
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'seaborn==0.13.2')

print('Dependencies installed successfully')

In [ ]:
# Standard imports
import json
import os
from collections import defaultdict, Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Set random seed for reproducibility
np.random.seed(42)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Imports loaded successfully')

In [ ]:
# Data loading with GitHub URL fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d0f06-curvature-discrepancy-for-citation-manip/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            print(f"Loaded data from GitHub URL")
            return data
    except Exception as e:
        print(f"GitHub URL failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            data = json.load(f)
            print(f"Loaded data from local file")
            return data
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined')

In [ ]:
# Load the demo data
data = load_data()

# Extract examples from the first dataset
dataset = data['datasets'][0]
examples = dataset['examples']

print(f"Dataset: {dataset['dataset']}")
print(f"Number of examples: {len(examples)}")
print(f"\nFirst example:")
print(json.dumps(examples[0], indent=2))

## Configuration

Set tunable parameters for the NLE method:
- `NEIGHBORHOOD_SIZE`: Number of hops to consider (1 = direct neighbors only)
- `NUM_SYNTHETIC_SUSPICIOUS`: Number of synthetic suspicious nodes to inject for validation
- `ENTROPY_BINS`: Number of bins for entropy histogram

In [ ]:
# Configuration - START WITH ABSOLUTE MINIMUM VALUES# These will be scaled up after verifying the notebook runs# Neighborhood size (hops) for NLE calculationNEIGHBORHOOD_SIZE = 1  # Direct neighbors only# Number of synthetic suspicious nodes to inject (for validation)NUM_SYNTHETIC_SUSPICIOUS = 5  # Increased for better validation# Entropy histogram binsENTROPY_BINS = 15  # Increased for finer histogram# Random seedRANDOM_SEED = 42print('Configuration set:')print(f'  NEIGHBORHOOD_SIZE = {NEIGHBORHOOD_SIZE}')print(f'  NUM_SYNTHETIC_SUSPICIOUS = {NUM_SYNTHETIC_SUSPICIOUS}')print(f'  ENTROPY_BINS = {ENTROPY_BINS}')

## Method: Neighborhood Label Entropy (NLE)

**Intuition**: In a normal citation network, papers cite works from diverse research areas. 
Suspicious patterns (citation rings, coordinated boosting) create neighborhoods with 
unusually low label diversity.

**Algorithm**:
1. For each node, gather labels of nodes within `NEIGHBORHOOD_SIZE` hops
2. Compute Shannon entropy of the label distribution
3. Low entropy = suspicious (potential citation manipulation)
4. High entropy = normal (diverse citations)

**Mathematical definition**:
```
NLE(node) = -Σ p(label) * log(p(label))
```
where p(label) is the empirical distribution of labels in the node's neighborhood.

In [ ]:
def build_graph_from_examples(examples):
    """Build adjacency list and node label mapping from examples."""
    adj = defaultdict(set)
    node_labels = {}
    
    for ex in examples:
        node_id = ex['metadata_node_id']
        label = ex['output']
        node_labels[node_id] = label
        
        # Parse neighbors from input
        input_data = json.loads(ex['input'])
        neighbors = input_data.get('neighbors', [])
        
        for nb in neighbors:
            adj[node_id].add(nb)
            adj[nb].add(node_id)  # Undirected
    
    return adj, node_labels

# Build graph
adj, node_labels = build_graph_from_examples(examples)

print(f'Graph built: {len(node_labels)} nodes, {sum(len(v) for v in adj.values())//2} edges')

In [ ]:
def compute_nle(node_id, adj, node_labels, neighborhood_size=1):
    """Compute Neighborhood Label Entropy for a node.
    
    Args:
        node_id: The node to compute NLE for
        adj: Adjacency list
        node_labels: Dict mapping node_id -> label
        neighborhood_size: Number of hops (currently only 1 supported)
    
    Returns:
        NLE score (float), or None if node has no neighbors with known labels
    """
    # Get direct neighbors (1-hop)
    neighbors = adj.get(node_id, set())
    
    # Collect labels of neighbors
    neighbor_labels = [node_labels.get(nb) for nb in neighbors if nb in node_labels]
    
    if not neighbor_labels:
        return None  # No labeled neighbors
    
    # Compute label distribution
    label_counts = Counter(neighbor_labels)
    total = len(neighbor_labels)
    
    # Compute Shannon entropy
    entropy = 0.0
    for count in label_counts.values():
        p = count / total
        if p > 0:
            entropy -= p * np.log(p)
    
    return entropy

# Compute NLE for all nodes
nle_scores = {}
for node_id in node_labels:
    nle = compute_nle(node_id, adj, node_labels, NEIGHBORHOOD_SIZE)
    if nle is not None:
        nle_scores[node_id] = nle

print(f'Computed NLE scores for {len(nle_scores)} nodes')
print(f'NLE range: [{min(nle_scores.values()):.4f}, {max(nle_scores.values()):.4f}]')
print(f'Mean NLE: {np.mean(list(nle_scores.values())):.4f}')
print(f'Std NLE: {np.std(list(nle_scores.values())):.4f}')

## Validation: Synthetic Suspicious Nodes

To validate that NLE can detect suspicious patterns, we inject synthetic nodes with:
- **Low entropy**: All neighbors have the same label (simulating citation ring)
- **High entropy**: Neighbors have diverse labels (normal pattern)

The method should assign low NLE scores to the suspicious nodes.

In [ ]:
def inject_suspicious_nodes(adj, node_labels, num_suspicious=3, random_seed=42):    """Inject synthetic suspicious nodes with low label diversity."""    np.random.seed(random_seed)        # Find max node ID to avoid collisions    max_id = max(node_labels.keys()) if node_labels else 0        synthetic_nodes = []    synthetic_adj = defaultdict(set)    synthetic_labels = dict(node_labels)        # Copy adjacency list    for k, v in adj.items():        synthetic_adj[k] = set(v)        for i in range(num_suspicious):        synth_id = max_id + 1 + i                # Pick a random existing label to dominate        all_labels = list(set(node_labels.values()))        dominant_label = np.random.choice(all_labels)                # Connect to 5-10 existing nodes        existing_nodes = list(node_labels.keys())        np.random.shuffle(existing_nodes)        num_connections = np.random.randint(5, 11)                neighbors = []        for nb in existing_nodes[:num_connections]:            neighbors.append(nb)            synthetic_adj[nb].add(synth_id)                synthetic_adj[synth_id] = set(neighbors)        # Give the synthetic node the dominant label        # Its neighbors have diverse labels (normal pattern), but we'll compute        # NLE using only direct neighbors which we can control        synthetic_labels[synth_id] = dominant_label                # For suspicious nodes: make all their neighbors have the same label        # by adding fake neighbor connections (not in the original graph)        # This simulates a citation ring        for nb in neighbors:            synthetic_labels[nb] = dominant_label  # Temporarily for NLE computation                synthetic_nodes.append(synth_id)        return synthetic_nodes, synthetic_adj, synthetic_labels# Inject synthetic nodessynth_nodes, synth_adj, synth_labels = inject_suspicious_nodes(    adj, node_labels, NUM_SYNTHETIC_SUSPICIOUS, RANDOM_SEED)print(f'Injected {len(synth_nodes)} synthetic suspicious nodes: {synth_nodes}')

In [ ]:
# Compute NLE for synthetic nodes
synth_nle = {}
for node_id in synth_nodes:
    nle = compute_nle(node_id, synth_adj, synth_labels, NEIGHBORHOOD_SIZE)
    synth_nle[node_id] = nle

print('NLE scores for synthetic nodes:')
for node_id, nle in synth_nle.items():
    print(f'  Node {node_id}: NLE = {nle:.4f}')

# Compare with normal nodes
normal_nle_values = list(nle_scores.values())
synth_nle_values = [v for v in synth_nle.values() if v is not None]

print(f'\nComparison:')
print(f'  Mean NLE (normal nodes): {np.mean(normal_nle_values):.4f}')
if synth_nle_values:
    print(f'  Mean NLE (synthetic suspicious): {np.mean(synth_nle_values):.4f}')
    print(f'  Suspicious nodes have lower NLE: {np.mean(synth_nle_values) < np.mean(normal_nle_values)}')

## Results Visualization

Visualize the distribution of NLE scores and highlight detected suspicious nodes.

In [ ]:
# Visualization: NLE distribution and detected suspicious nodesfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Histogram of NLE scoresax1 = axes[0]normal_scores = list(nle_scores.values())ax1.hist(normal_scores, bins=ENTROPY_BINS, alpha=0.7, color='blue', edgecolor='black', label='Normal nodes')# Add synthetic nodes if availablesynth_scores = [v for v in synth_nle.values() if v is not None]if synth_scores:    ax1.hist(synth_scores, bins=5, alpha=0.7, color='red', edgecolor='black', label='Synthetic suspicious')ax1.set_xlabel('Neighborhood Label Entropy (NLE)')ax1.set_ylabel('Count')ax1.set_title('Distribution of NLE Scores')ax1.legend()ax1.grid(True, alpha=0.3)# Box plot comparing normal vs suspiciousax2 = axes[1]if synth_scores:    data_to_plot = [normal_scores, synth_scores]    ax2.boxplot(data_to_plot)    ax2.set_xticklabels(['Normal', 'Suspicious'])    ax2.set_ylabel('NLE Score')    ax2.set_title('NLE: Normal vs Suspicious Nodes')else:    ax2.boxplot([normal_scores])    ax2.set_ylabel('NLE Score')    ax2.set_title('NLE Distribution (Normal Nodes Only)')ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()# Print summary tableprint('\n' + '='*60)print('SUMMARY: Neighborhood Label Entropy (NLE) Detection Results')print('='*60)print(f'{'Metric':<30} {'Value':>15}')print('-'*60)print(f'{'Total nodes analyzed':<30} {len(nle_scores):>15}')print(f'{'Mean NLE (normal)':<30} {np.mean(normal_scores):>15.4f}')print(f'{'Std NLE (normal)':<30} {np.std(normal_scores):>15.4f}')if synth_scores:    print(f'{'Mean NLE (suspicious)':<30} {np.mean(synth_scores):>15.4f}')    print(f'{'Detection success':<30} {'Yes' if np.mean(synth_scores) < np.mean(normal_scores) else 'No':>15}')print('='*60)